# Contexta Research — 04 Sense Annotation Improved

Built on SenseForge shared modules. Run sequentially from the repository root or notebooks directory. Missing data is explicitly skipped; no fabricated results.

In [2]:
from pathlib import Path
import os
import sys
import json

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / "src"))

DATA_ROOT = Path("/Users/suvratsmacbookair/MyLocalFiles/NLP project")
os.environ["CONTEXTA_DATA_ROOT"] = str(DATA_ROOT)
os.environ["SENSEFORGE_DATA_ROOT"] = str(DATA_ROOT)

ART = ROOT / "artifacts"
ART.mkdir(parents=True, exist_ok=True)

print("Project:", ROOT)
print("Data:", DATA_ROOT)

Project: /Users/suvratsmacbookair/MyLocalFiles/NLP project/SenseForge
Data: /Users/suvratsmacbookair/MyLocalFiles/NLP project


## Persistent annotation and explicit adjudication

In [4]:
from senseforge.annotation import save,records,gold,adjudicate,adjudicated_gold
from senseforge.research import read_jsonl,write_jsonl,grouped_split
queue=read_jsonl(ART/'annotation_queue.jsonl')
print('Pending queue:',len(queue),'Existing annotations:',len(records()))
if queue: print('Example:',{k:queue[0][k] for k in ('id','text','word')})
print('Use save(item_id,word,sense_id,annotator) for independent human reviews; adjudicate(...) for disputes.')

Pending queue: 448664 Existing annotations: 0
Example: {'id': '12_34', 'text': 'Browse the various methods of the current accessible', 'word': 'current'}
Use save(item_id,word,sense_id,annotator) for independent human reviews; adjudicate(...) for disputes.


## Export human-only gold and group-disjoint splits

In [6]:
reviewed={r['item_id']:r['sense_id'] for r in gold()}
reviewed.update({r['item_id']:r['sense_id'] for r in adjudicated_gold()})
verified=[{**r,'sense_id':reviewed[r['id']]} for r in queue if r['id'] in reviewed]
parts=grouped_split(verified)
for split,rows in parts.items():print(split,write_jsonl(ART/f'wsd_{split}.jsonl',rows))
print('No synthetic labels generated.')

train 0
validation 0
test 0
No synthetic labels generated.


In [10]:
import json
import random
from collections import defaultdict

random.seed(42)

source = ART / "annotation_queue.jsonl"
sample_path = ART / "annotation_sample.jsonl"

MAX_PER_WORD = 30
samples = defaultdict(list)
counts = defaultdict(int)

with source.open(encoding="utf-8") as f:
    for line in f:
        row = json.loads(line)
        word = row["word"]
        counts[word] += 1

        bucket = samples[word]
        if len(bucket) < MAX_PER_WORD:
            bucket.append(row)
        else:
            j = random.randrange(counts[word])
            if j < MAX_PER_WORD:
                bucket[j] = row

selected = [
    row
    for word in sorted(samples)
    for row in samples[word]
]

random.shuffle(selected)

with sample_path.open("w", encoding="utf-8") as f:
    for row in selected:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print("Total candidates:", sum(counts.values()))
print("Selected for annotation:", len(selected))
print("Saved:", sample_path)

Total candidates: 448664
Selected for annotation: 1050
Saved: /Users/suvratsmacbookair/MyLocalFiles/NLP project/SenseForge/artifacts/annotation_sample.jsonl


In [12]:
from senseforge.core import INVENTORY

print("Total ambiguous words:", len(INVENTORY))

for word in list(INVENTORY)[:5]:
    print("\n", word)
    print(INVENTORY[word])

Total ambiguous words: 35

 bank
{'lemma': 'bank', 'pos': 'ANY', 'senses': [{'id': 'bank.1', 'definition': 'financial institution', 'hindi': 'बैंक', 'cues': ['money', 'deposit', 'loan', 'account', 'finance'], 'priority': 1}, {'id': 'bank.2', 'definition': 'river shore water stream nadi नदी', 'hindi': 'किनारा', 'cues': ['river', 'shore', 'water', 'stream', 'nadi', 'नदी'], 'priority': 1}], 'default': 'bank.1', 'provenance': 'curated initial inventory; requires linguistic review'}

 bat
{'lemma': 'bat', 'pos': 'ANY', 'senses': [{'id': 'bat.1', 'definition': 'sports equipment', 'hindi': 'बल्ला', 'cues': ['cricket', 'ball', 'hit', 'match'], 'priority': 1}, {'id': 'bat.2', 'definition': 'flying cave nocturnal animal', 'hindi': 'चमगादड़', 'cues': ['flying', 'cave', 'nocturnal', 'animal'], 'priority': 1}], 'default': 'bat.1', 'provenance': 'curated initial inventory; requires linguistic review'}

 bark
{'lemma': 'bark', 'pos': 'ANY', 'senses': [{'id': 'bark.1', 'definition': 'tree covering', '

In [22]:
import json
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from senseforge.annotation import save, records
from senseforge.core import INVENTORY
sample_path = ART / "annotation_pilot.jsonl"

with sample_path.open(encoding="utf-8") as f:
    items = [json.loads(line) for line in f]

annotator = widgets.Text(
    value="annotator_1",
    description="Annotator:",
    style={"description_width": "initial"}
)

position = widgets.BoundedIntText(
    value=1, min=1, max=len(items),
    description="Record:",
    layout=widgets.Layout(width="180px")
)

sense_menu = widgets.Dropdown(
    description="Sense:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="550px")
)

previous = widgets.Button(description="Previous")
save_next = widgets.Button(description="Save & Next", button_style="success")
skip = widgets.Button(description="Skip")
refresh = widgets.Button(description="Refresh")

sentence_view = widgets.HTML()
status = widgets.HTML()
message = widgets.Output()

def show_record():
    row = items[position.value - 1]
    word = row["word"]
    entry = INVENTORY.get(word)

    if not entry:
        sense_menu.options = [("Unknown word", "uncertain")]
    else:
        sense_menu.options = [
            (
                f"{s['id']} — {s['definition']} → {s['hindi']}",
                s["id"]
            )
            for s in entry["senses"]
        ]
        sense_menu.options = list(sense_menu.options) + [
            ("Uncertain", "uncertain"),
            ("Other", "other"),
            ("Trash", "trash")
        ]

    text = row["text"]
    start, end = row["start"], row["end"]

    from html import escape
    highlighted = (
        escape(text[:start])
        + "<mark>" + escape(text[start:end]) + "</mark>"
        + escape(text[end:])
    )

    sentence_view.value = (
        f"<h3>{highlighted}</h3>"
        f"<p><b>Hindi:</b> {escape(row.get('hindi_sentence', ''))}</p>"
        f"<p><b>Target:</b> {escape(word)}</p>"
    )

    status.value = (
        f"<b>Record {position.value} / {len(items)}</b>"
    )

def move(step):
    position.value = max(1, min(len(items), position.value + step))
    show_record()

def on_save(_):
    row = items[position.value - 1]

    with message:
        clear_output()
        try:
            save(
                row["id"],
                row["word"],
                sense_menu.value,
                annotator.value.strip()
            )
            print("Saved:", row["id"], sense_menu.value)
            move(1)
        except Exception as e:
            print("Error:", e)

def on_previous(_):
    move(-1)

def on_skip(_):
    move(1)

def on_refresh(_):
    show_record()

def on_position(change):
    if change["name"] == "value":
        show_record()

previous.on_click(on_previous)
save_next.on_click(on_save)
skip.on_click(on_skip)
refresh.on_click(on_refresh)
position.observe(on_position, names="value")

display(
    widgets.VBox([
        widgets.HBox([annotator, position]),
        status,
        sentence_view,
        sense_menu,
        widgets.HBox([previous, save_next, skip, refresh]),
        message
    ])
)

show_record()

In [16]:
from senseforge.annotation import records

saved = records()
print("Total saved annotations:", len(saved))
print("Latest annotation:", saved[-1] if saved else None)

Total saved annotations: 1
Latest annotation: {'item_id': '4677852_33', 'annotator': 'annotator_1', 'sense_id': 'train.1', 'status': 'reviewed', 'timestamp': '2026-11-16T17:05:05.499545+00:00'}


In [18]:
import json
from collections import defaultdict

pilot_path = ART / "annotation_pilot.jsonl"
pilot = defaultdict(list)

for row in items:
    word = row["word"]
    if len(pilot[word]) < 5:
        pilot[word].append(row)

pilot_items = [
    row
    for word in sorted(pilot)
    for row in pilot[word]
]

with pilot_path.open("w", encoding="utf-8") as f:
    for row in pilot_items:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print("Pilot records:", len(pilot_items))
print("Words covered:", len(pilot))
print("Saved:", pilot_path)

Pilot records: 175
Words covered: 35
Saved: /Users/suvratsmacbookair/MyLocalFiles/NLP project/SenseForge/artifacts/annotation_pilot.jsonl
